# Data Ingestion — Getting Started

The ingestion pipeline is the primary entry point for loading geospatial data into
DynaStore collections. It is exposed as an **OGC Process** (Part 1) and follows a
fully asynchronous, task-based architecture.

## How it works

```
POST /catalogs/{cat}/collections/{coll}/processes/ingestion/execution
  │
  ├─ 1. Validate: collection exists, has a physical LayerConfig
  ├─ 2. Create task record (PENDING) in tasks.tasks
  ├─ 3. Return 202 Accepted with status_url for polling
  │
  Dispatcher claims task (FOR UPDATE SKIP LOCKED)
  │
  ├─ 4. Resolve or create asset from URI / asset_id
  ├─ 5. Open source file (Fiona) and stream records in batches
  ├─ 6. Map columns → geometry + attributes + identity
  ├─ 7. Upsert batches to all WRITE-routed drivers
  ├─ 8. Recalculate collection extents
  └─ 9. Mark task COMPLETED (or FAILED)
```

## Task lifecycle

| Status | Meaning |
|--------|---------|
| `PENDING` | Queued, waiting for a runner to claim |
| `ACTIVE` | Claimed by a runner, heartbeat expected every 30s |
| `COMPLETED` | Successfully finished |
| `FAILED` | Error encountered, may retry with exponential backoff |
| `DEAD_LETTER` | Max retries exhausted, requires manual intervention |

This notebook demonstrates the end-to-end flow: create a collection,
submit an ingestion request, monitor progress, and verify results.

In [ ]:
# === Parameters ===
# Adjust these to point at your DynaStore instance.
# For the on-premise 'catalog' service (no auth required), use port 82.

BASE = "http://localhost:8000"   # API base URL
CATALOG_ID = "demo-ingestion"    # Will be created by this notebook
COLLECTION_ID = "crop-stations"  # Target collection for ingestion

In [ ]:
import httpx
import asyncio
import json

client = httpx.AsyncClient(base_url=BASE, timeout=60)

# Create demo catalog
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Ingestion Demo"})
print("Catalog:", r.status_code)

# Create collection with a physical layer config (required for ingestion)
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Crop Monitoring Stations",
    "description": "Demo collection for ingestion tutorial",
    "license": "CC-BY-4.0",
})
print("Collection:", r.status_code)

---
## 1. CSV Ingestion with lat/lon columns

The most common scenario: a CSV file with latitude and longitude columns.
The `column_mapping` tells the ingestion engine how to extract geometry
from the source columns.

### Key fields in `column_mapping`

| Field | Purpose |
|-------|---------|
| `external_id` | Source column used as unique identity for conflict resolution |
| `csv_lat_column` | Column containing latitude |
| `csv_lon_column` | Column containing longitude |
| `csv_elevation_column` | Optional: column with elevation (creates 3D points) |
| `attributes_source_type` | `"all"` (default) or `"explicit_list"` |

The ingestion endpoint is an OGC Process:

```
POST /catalogs/{catalog_id}/collections/{collection_id}/processes/ingestion/execution
```

In [ ]:
# Submit an ingestion request for a CSV file with lat/lon columns.
# In production, the URI points to a real file (local path or gs:// bucket).
# For this demo, we show the request structure.

ingestion_body = {
    "inputs": {
        "catalog_id": CATALOG_ID,
        "collection_id": COLLECTION_ID,
        "ingestion_request": {
            "asset": {
                "uri": "/data/crop_stations.csv"
            },
            "encoding": "utf-8",
            "column_mapping": {
                "external_id": "station_id",
                "csv_lat_column": "latitude",
                "csv_lon_column": "longitude",
                "csv_elevation_column": "elevation_m",
                "attributes_source_type": "all"
            },
            "source_srid": 4326
        }
    }
}

print("Ingestion request body:")
print(json.dumps(ingestion_body, indent=2))

# To actually submit (requires a real file at the URI):
# r = await client.post(
#     f"/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution",
#     json=ingestion_body,
# )
# print("Status:", r.status_code)  # 201 (sync) or 202 (async)
# if r.status_code == 202:
#     task = r.json()
#     print("Task ID:", task["jobID"])
#     print("Status URL:", task.get("links", [{}])[0].get("href"))

---
## 2. Inline feature ingestion (no file needed)

For testing and small datasets, you can ingest features directly via the
STAC/OGC Features API without going through the task system.
This is synchronous and immediate.

In [ ]:
# Direct feature ingestion via STAC Items endpoint (synchronous)
features = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": "station-001",
            "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
            "properties": {
                "name": "Roma Fiumicino",
                "crop_type": "wheat",
                "yield_t_ha": 4.2,
                "year": 2024
            }
        },
        {
            "type": "Feature",
            "id": "station-002",
            "geometry": {"type": "Point", "coordinates": [11.25, 43.77]},
            "properties": {
                "name": "Firenze Peretola",
                "crop_type": "olive",
                "yield_t_ha": 2.8,
                "year": 2024
            }
        },
        {
            "type": "Feature",
            "id": "station-003",
            "geometry": {"type": "Point", "coordinates": [9.19, 45.46]},
            "properties": {
                "name": "Milano Linate",
                "crop_type": "rice",
                "yield_t_ha": 6.1,
                "year": 2024
            }
        }
    ]
}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=features,
)
print("Ingest:", r.status_code)

---
## 3. Column mapping modes

The `column_mapping.attributes_source_type` controls which source columns
become feature properties.

### `"all"` (default)
All source columns except geometry and identity columns become properties.
Quick and easy for exploratory work.

### `"explicit_list"`
Only columns listed in `attribute_mapping` are included. Each entry maps
a `source` column to a `map_to` target name. You can also inject `constant`
values (e.g., a fixed data source tag).

```json
{
    "column_mapping": {
        "external_id": "station_id",
        "csv_lat_column": "lat",
        "csv_lon_column": "lon",
        "attributes_source_type": "explicit_list",
        "attribute_mapping": [
            {"source": "station_name", "map_to": "name"},
            {"source": "crop",         "map_to": "crop_type"},
            {"source": "yield_kg",     "map_to": "yield_t_ha"},
            {"constant": "FAO",         "map_to": "data_source"}
        ]
    }
}
```

Use `explicit_list` when:
- Source columns have inconsistent names across files
- You need to rename columns to match your schema
- You want to inject constant metadata values
- You need to exclude sensitive or irrelevant columns

---
## 4. Task monitoring

When ingestion runs asynchronously (the default for file-based ingestion),
you get back a 202 response with a task ID. Poll the task status endpoint
to track progress.

The task moves through: `PENDING` → `ACTIVE` → `COMPLETED` (or `FAILED`).
While `ACTIVE`, the `progress` field reports the percentage of records processed.

In [ ]:
# Poll task status (use a real task_id from an async ingestion response)
# task_id = "<from 202 response>"

async def poll_task(catalog_id: str, task_id: str, interval: int = 2, max_wait: int = 120):
    """Poll a task until terminal state."""
    elapsed = 0
    while elapsed < max_wait:
        r = await client.get(f"/tasks/catalogs/{catalog_id}/{task_id}")
        if r.status_code != 200:
            print(f"Error polling task: {r.status_code}")
            return None
        task = r.json()
        status = task.get("status", "unknown")
        progress = task.get("progress", 0)
        print(f"  [{elapsed}s] status={status} progress={progress}%")
        if status in ("successful", "failed", "dismissed"):
            return task
        await asyncio.sleep(interval)
        elapsed += interval
    print("Timeout waiting for task completion")
    return None

# Example usage:
# result = await poll_task(CATALOG_ID, task_id)
# print(json.dumps(result, indent=2))
print("poll_task() defined — use with a real task_id from an async ingestion")

---
## 5. Verify ingested data

After ingestion completes, query the collection via the STAC Items endpoint
to confirm features were stored correctly.

In [ ]:
# List items in the collection
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items")
items = r.json()

print(f"Total features: {items.get('numberMatched', len(items.get('features', [])))}")
for feat in items.get("features", []):
    props = feat.get("properties", {})
    geom = feat.get("geometry", {})
    coords = geom.get("coordinates", [])
    print(f"  {feat.get('id')}: {props.get('name')} @ {coords} — {props.get('crop_type')} {props.get('yield_t_ha')} t/ha")

---
## 6. Batch tuning

For large files, tune the batch parameters to balance memory usage and throughput.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `database_batch_size` | 500 | Records per database transaction |
| `max_batch_memory_mb` | 100 | Memory threshold before flushing (takes precedence) |
| `read_batch_size` | 1000 | Records to read from source per chunk |
| `offset` | 0 | Skip N records from the start |
| `limit` | None | Process at most N records |

### Guidelines

- **Small files (<10K records)**: defaults work fine
- **Large files (>100K records)**: increase `database_batch_size` to 2000–5000
- **Memory-constrained**: lower `max_batch_memory_mb` to 50
- **Resume after failure**: use `offset` to skip already-ingested records
- **Sampling**: use `limit` to ingest a subset for testing

```json
{
    "ingestion_request": {
        "database_batch_size": 2000,
        "max_batch_memory_mb": 50,
        "read_batch_size": 5000,
        "offset": 0,
        "limit": 10000
    }
}
```

---
## 7. Geometry formats

The ingestion engine supports multiple geometry input formats:

| Format | Column mapping field | Use case |
|--------|---------------------|----------|
| Lat/Lon columns | `csv_lat_column` + `csv_lon_column` | CSV with separate coordinate columns |
| WKT | `csv_wkt_column` | CSV with Well-Known Text geometry |
| WKB | `geometry_wkb` | Parquet/binary with Well-Known Binary |
| Native GeoJSON | (automatic) | GeoJSON/Shapefile via Fiona — geometry extracted automatically |

### WKT example

```json
{
    "column_mapping": {
        "external_id": "parcel_id",
        "csv_wkt_column": "geom_wkt",
        "attributes_source_type": "all"
    }
}
```

### GeoJSON/Shapefile

For files that Fiona reads natively (`.geojson`, `.shp`, `.gpkg`), geometry
is extracted automatically from the `geometry` field. No column mapping needed
for geometry — just set `external_id` and attributes.

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print("Cleanup:", r.status_code)
await client.aclose()